- 原始数据 → Pydantic 模型     反序列化/解析
- Pydantic 模型 → Python 字典  model_dump
- Pydantic 模型 → JSON 字符串 model_dump_json
- model_validate()      → 接收 Python 字典或对象
- model_validate_json() → 接收 JSON 字符串或 bytes

### 从字典创建模型

In [ ]:
from pydantic import BaseModel


class UserDTO(BaseModel):
    id: int
    name: str


user = UserDTO.model_validate(
    {
        "id": "10",
        "name": "Alice",
    }
)
"""
Pydantic 默认会尝试转换:
user.id == 10
type(user.id) is int

"""

# 也可以直接
user2 = UserDTO(
    id="10",
    name="Alice",
)

### 从 JSON 字符串创建模型
- model_validate()      → 接收 Python 字典或对象
- model_validate_json() → 接收 JSON 字符串或 bytes

In [ ]:
json_text = """
{
  "id": 10,
  "name": "Alice"
}
"""


user = UserDTO.model_validate_json(json_text) # 不要把 Python 字典直接传


### Python 模式与 JSON 模式
考虑时间字段

In [ ]:
from datetime import datetime, timezone


class EventDTO(BaseModel):
    created_at: datetime


event = EventDTO(
    created_at=datetime.now(timezone.utc)
)

In [ ]:
# 普通字典
event.model_dump() # created_at还是py对象datetime
# json兼容字典
event.model_dump(mode="json") # 结果中的时间会转换成字符串
# json对象
event.model_dump_json() # 结果中的时间会转换成字符串

### 别名
```
model_dump(
    mode="json",
    by_alias=True,
)
```
`by_alias=True`按别名输出，不填默认按字段名输出

In [ ]:
from datetime import datetime

from pydantic import BaseModel, ConfigDict, Field


class UserResponseDTO(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )

    user_id: int = Field(alias="userId")
    created_at: datetime = Field(alias="createdAt")

In [ ]:
user = UserResponseDTO.model_validate(
    {
        "userId": 10,
        "createdAt": "2026-10-02T10:00:00Z",
    }
)
user.user_id == 10 # Python内部仍然使用 user_id
# 按字段名输出
user.model_dump(mode="json") # 结果中使用的是下划线
# 按别名输出
user.model_dump(
    mode="json",
    by_alias=True,
)

### populate_by_name=True
配置`ConfigDict(populate_by_name=True)`，外部可以按字段名输入。

在内部代码、测试和渐进迁移中比较方便

### 批量生成驼峰别名
使用别名生成器

In [5]:
from pydantic import BaseModel, ConfigDict
from pydantic.alias_generators import to_camel
from datetime import datetime, timezone
from rich import print  

class UserResponseDTO(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel, # 生成驼峰别名
        populate_by_name=True, #允许按字段名输入
    )

    user_id: int
    display_name: str
    created_at: datetime

user = UserResponseDTO(
    userId=10,
    displayName="Alice",
    createdAt=datetime.now(timezone.utc),
)

print(user)

user.model_dump(
    mode="json",
    by_alias=True,
)

UserResponseDTO(
    user_id=10,
    display_name='Alice',
    created_at=datetime.datetime(2026, 10, 2, 15, 40, 16, 732646, tzinfo=datetime.timezone.utc)
)

{'userId': 10,
 'displayName': 'Alice',
 'createdAt': '2026-10-02T15:40:16.732646Z'}

### 过滤输出字段
- 忽略 None：`model.model_dump(exclude_none=True)`
- 忽略客户端未提供的字段：`model.model_dump(exclude_unset=True)`
- 忽略等于默认值的字段：`model.model_dump(exclude_defaults=True)`